In [1]:
from transformers import AutoTokenizer
from modeling_xmistral import XMistralForCausalLM
from datasets import load_from_disk

In [2]:
import torch

In [3]:
XRAG_TOKEN = "<xRAG>" 

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:

model_name = "Hannibal046/xrag-7b"
model = XMistralForCausalLM.from_pretrained(
    model_name,
    torch_dtype = torch.bfloat16,
    low_cpu_mem_usage = True,
    )
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    add_eos_token=False,
    use_fast=False,
    padding_side='left'
    )
model.set_xrag_token_id(tokenizer.convert_tokens_to_ids(XRAG_TOKEN))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
original_sample_data = load_from_disk("data/sample_data")
for item in original_sample_data:
    print(item)

{'gem_id': 'gem-squad_v2-train-21300', 'id': '56e6c52c6fe0821900b8eb67', 'title': 'Adult_contemporary_music', 'context': 'Hard rock had been established as a mainstream genre by 1965. From the end of the 1960s, it became common to divide mainstream rock music into soft and hard rock, with both emerging as major radio formats in the US. Soft rock was often derived from folk rock, using acoustic instruments and putting more emphasis on melody and harmonies. Major artists included Barbra Streisand, Carole King, Cat Stevens, James Taylor and Bread.', 'question': 'Along with soft rock, what type of music made up mainstream rock music in the late 1960s?', 'target': 'Along with soft rock, what type of music made up mainstream rock music in the late 1960s?', 'references': ['Along with soft rock, what type of music made up mainstream rock music in the late 1960s?'], 'answers': {'text': ['hard rock'], 'answer_start': [152]}}
{'gem_id': 'gem-squad_v2-train-29034', 'id': '5ad379e5604f3c001a3fe3be'

In [7]:
embeded_docs = torch.load("tensor_files/embeded_sample.pt")
relevant_doc = embeded_docs[0].to(device)

In [14]:
rag_template = """[INST] Background: {xrag_token}, which also means: [/INST]"""
prompt = rag_template.format_map(dict(xrag_token=XRAG_TOKEN))
print(prompt)

[INST] Background: <xRAG>, which also means: [/INST]


In [19]:
tokenized_prompt = tokenizer(prompt, return_tensors="pt").to(device)

In [20]:
tokenized_prompt

{'input_ids': tensor([[    1,   733, 16289, 28793, 24316, 28747, 28705, 32001,  1200,   690,
           835,  2825, 28747,   733, 28748, 16289, 28793]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [25]:
output = model.generate(
        **tokenized_prompt,
        do_sample=False,
        max_new_tokens=100,
        retrieval_embeds = relevant_doc.unsqueeze(0),
    )

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Shape of final inputs_embeds fed to the model: torch.Size([1, 17, 4096])


In [26]:
tokenizer.decode(output[0])

'1. Soft rock, which is a subgenre of rock music that originated in the United States and the United Kingdom during the mid-1960s. It is characterized by a more melodic and mellow sound than traditional rock music. Soft rock is often associated with the singer-songwriter movement and the folk rock genre.\n\n2. Hard rock, which is a subgenre of rock music that emerged in the late 1960s and early'